# 09 — Modelado predictivo (Objetivo 2)
### TFM · Análisis multidimensional del estrés estudiantil

Objetivo 2 de la propuesta: *desarrollar modelos de IA para la predicción del estrés, incluyendo clasificación sobre datos estructurados y NLP sobre datos textuales, manteniendo un análisis independiente por dataset.* En el Alcance del máster, el Machine Learning es la parte **opcional** de la Entrega 3.

---
## Decisiones de diseño

| Decisión | Qué se hizo | Por qué |
|---|---|---|
| **Fuentes** | 9 fuentes con `uso_modelado='incluido'` → **8 tareas** (Dreaddit train+test forman una sola tarea NLP). Los 3 datasets sintéticos quedan fuera | Exigencia del tutor: modelar datos sintéticos resta validez científica |
| **Estrella** | La política de modelado se **lee de `vw_dim_dataset`** (columna `uso_modelado`) y los resultados se **cargan al DWH** (`dwh_modelo_resultados`), enlazados por `id_dataset` | El modelado queda integrado en la arquitectura aprobada y Power BI puede filtrar registros y resultados con la misma dimensión |
| **Independencia** | Un modelo por dataset, sin fusionar fuentes | Metodología de análisis en paralelo |
| **Candidatos** | Baseline + Regresión Logística + Random Forest + XGBoost. En NLP: además TF-IDF con LR / SVM / Naive Bayes y un modelo híbrido | Días 6 y 8 |
| **Evaluación** | n < 1000: validación cruzada 5×3 · n ≥ 1000: hold-out 80/20 estratificado · Dreaddit: partición original de los autores | Con 87 filas un único hold-out es inestable (Día 5: sobreajuste) |
| **Métrica principal** | **AUC**, junto con F1 y Accuracy comparados contra el baseline | Día 5: el Accuracy engaña con clases desbalanceadas |
| **Fuga de datos** | Se excluyen las variables que construyen el target (ítems GAD-7, ítems DASS, `anxiety` en MH Survey) | Evita resultados triviales |
| **Ética** | Religión, orientación sexual y raza no se usan como predictores | No se modelan atributos sensibles |
| **Deep learning** | MLP / LSTM (Días 9-10) no se aplican: con datos tabulares de 87 a 54 000 filas, regresión y árboles son el punto de partida estándar; queda como línea futura | — |
| **Industrialización** | Cada pipeline completo (preprocesado + modelo) se guarda en `.pkl` + `.json` | Día 11 |

### Tipo de problema de cada tarea
| Tarea | Tipo | Target | Clases | Balance | Métrica principal |
|---|---|---|---|---|---|
| B1_ACADSTRESS | Multiclase | Índice de estrés 1-5 | 5 | Desbalanceado (clases 1-2 < 11 %) | F1-macro + AUC |
| B1_ANXIETY | Binaria | GAD-7 ≥ 10 | 2 | Desbalanceado (83 / 17 %) | AUC + F1 |
| B1_MHSURVEY | Multiclase | Depresión Bajo / Moderado / Alto | 3 | Moderado (31 / 24 / 45 %) | F1-macro + AUC |
| B2_SMMH | Binaria | "Se siente deprimido/a" ≥ 4 | 2 | Casi balanceado (54 / 46 %) | AUC + F1 |
| B3_DEPSTUDENT | Binaria | Depresión Sí / No | 2 | Balanceado (50 / 50 %) | AUC + F1 |
| B4_DASS | Multiclase | Estrés Bajo / Moderado / Alto (baremos) | 3 | Casi balanceado (29 / 36 / 36 %) | F1-macro + AUC |
| B4_MENTALHEALTH | Binaria | Ha buscado tratamiento | 2 | Balanceado (51 / 49 %) | AUC + F1 |
| B3_DREADDIT (NLP) | Binaria | Estrés 0 / 1 | 2 | Casi balanceado (48 / 52 %, prueba) | AUC + F1 + Accuracy |

El tipo de problema decide la métrica: con clases desbalanceadas el Accuracy engaña (en Anxiety, predecir siempre "no alto" da un 83 % de Accuracy con F1 = 0), por eso la métrica principal es siempre el AUC acompañado del F1.

**Cómo leer las tablas:** `brecha_acc` = Accuracy de entrenamiento − Accuracy de prueba. Valores superiores a ≈0.15 indican **sobreajuste** (el modelo memoriza). Las cifras citadas en los análisis corresponden a una ejecución con semilla 42; pueden variar en milésimas según la versión de las librerías.

### Paso 0 — Instalar librerías (scikit-learn, joblib, XGBoost y conector MySQL)

In [ ]:
%pip install scikit-learn joblib xgboost sqlalchemy pymysql --quiet
print("Si es la primera instalacion, reinicia el kernel (Kernel > Restart) y continua.")

### Paso 1 — Librerías
Si `xgboost` no estuviera disponible, el notebook usa automáticamente `HistGradientBoosting` de scikit-learn (misma familia de algoritmos).

In [ ]:
import os, re, json, warnings, joblib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_validate, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sqlalchemy import create_engine, text, inspect

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.95)

try:
    from xgboost import XGBClassifier
    BOOST_NAME = "XGBoost"
except ImportError:
    BOOST_NAME = "Gradient Boosting (HistGB)"   # alternativa de scikit-learn si no hay xgboost
BASE_NAME = "Baseline (clase mayoritaria)"
print("Modelo de boosting en uso:", BOOST_NAME)

### Paso 2 — Rutas y semilla
Mismo esquema `BASE` / `ALMC` del resto del proyecto. El notebook intenta conectarse a la estrella (`tfm_estres_estudiantil_2`, XAMPP en verde); **si no hay conexión, modela igualmente desde los CSV** con la política por defecto. Los modelos se guardan en `...\Objetivo2\modelos` y las figuras en `...\Objetivo2\figuras_modelado`. La semilla fija (42) hace los resultados reproducibles.

In [ ]:
# -- Rutas (mismo esquema BASE / ALMC del resto del proyecto) -------------------
BASE = r"../data/raw"
ALMC = r"../salidas/objetivo2"

FIG = os.path.join(ALMC, "figuras_modelado")
MOD = os.path.join(ALMC, "modelos")
os.makedirs(FIG, exist_ok=True)
os.makedirs(MOD, exist_ok=True)

SEED = 42
EJECUCION = datetime.now()
RESULTADOS, META = [], []      # acumuladores para el resumen final
print("Ejecucion:", EJECUCION.strftime("%Y-%m-%d %H:%M:%S"), "| semilla:", SEED)

# -- Conexion a la estrella (opcional): la politica de modelado se lee de vw_dim_dataset ---------
DB_USER, DB_PASS, DB_HOST, DB_NAME = "root", "", "localhost", "tfm_estres_estudiantil_2"
TAREA_DATASET = {   # tarea de modelado -> id_dataset de dim_dataset (Dreaddit train+test = ids 8 y 9; se registra con el 8)
    "B1_ACADSTRESS": 1, "B1_ANXIETY": 2, "B1_MHSURVEY": 3, "B2_SMMH": 6,
    "B3_DREADDIT": 8, "B3_DEPSTUDENT": 10, "B4_DASS": 11, "B4_MENTALHEALTH": 12}
engine, DS_ESTRELLA = None, None
try:
    engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")
    DS_ESTRELLA = pd.read_sql(text("SELECT id_dataset, codigo, dataset, bloque, calidad, uso_modelado "
                                   "FROM vw_dim_dataset ORDER BY id_dataset"), engine)
    print("Conectado a la estrella:", DB_NAME)
except Exception as e:
    engine, DS_ESTRELLA = None, None
    print("Sin conexion a MySQL (", type(e).__name__, "): se modela igualmente desde los CSV con la politica por defecto.")

def permitido(tarea):
    """Politica de modelado leida de la estrella: solo fuentes con uso_modelado = 'incluido'."""
    if DS_ESTRELLA is None:
        return True
    fila = DS_ESTRELLA[DS_ESTRELLA.id_dataset == TAREA_DATASET[tarea]]
    return (not fila.empty) and fila.uso_modelado.iloc[0] == "incluido"

### Paso 2b — Política de modelado leída de la estrella
La tabla muestra los 12 datasets de `vw_dim_dataset` y cuáles se modelan. La columna `uso_modelado` es el **único origen de la política**: si una fuente pasa a 'excluido' en la base de datos, este notebook la omite sin tocar el código. Dreaddit ocupa dos filas (train = 8, test = 9) pero es una única tarea.

In [ ]:
# Politica de modelado leida de la ESTRELLA (vw_dim_dataset): solo se modelan las fuentes con uso_modelado = 'incluido'
if DS_ESTRELLA is not None:
    ids_tareas = set(TAREA_DATASET.values()) | {9}      # Dreaddit test (id 9) se evalua dentro de la tarea B3_DREADDIT
    vista = DS_ESTRELLA.assign(se_modela=lambda t: np.where(t.id_dataset.isin(ids_tareas), "SI", "no"))
    display(vista[["id_dataset", "codigo", "bloque", "calidad", "uso_modelado", "se_modela"]])
    incoherentes = vista[(vista.se_modela == "SI") & (vista.uso_modelado != "incluido")]
    print("Fuentes en la lista de modelado pero 'excluidas' en la estrella (se omitiran):", len(incoherentes))
else:
    print("Estrella no disponible: se aplica la politica por defecto (8 tareas sobre las 9 fuentes incluidas).")

### Paso 3 — Funciones de modelado
- `crear_modelos`: el baseline (predice siempre la clase mayoritaria) fija el **suelo** que cualquier modelo debe superar.
- `crear_preprocesador`: numéricas → mediana + estandarizado; categóricas → moda + **One-Hot** (variables sin orden natural, Día 8).
- `metricas`: Accuracy, F1 y AUC (en multiclase: F1-macro y AUC *one-vs-rest*).

In [ ]:
def crear_modelos(multiclase):
    """Cuatro candidatos por dataset: baseline + 3 algoritmos vistos en clase."""
    if BOOST_NAME == "XGBoost":
        boost = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, random_state=SEED, n_jobs=-1,
                              eval_metric="mlogloss" if multiclase else "logloss")
    else:
        boost = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300,
                                               random_state=SEED)
    return {
        BASE_NAME: DummyClassifier(strategy="most_frequent"),
        "Regresion Logistica": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                                class_weight="balanced", random_state=SEED, n_jobs=-1),
        BOOST_NAME: boost,
    }

def crear_preprocesador(num_cols, cat_cols):
    """Numericas: mediana + estandarizado. Categoricas: moda + One-Hot (sin orden natural)."""
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    trans = [("num", num, num_cols)]
    if cat_cols:
        trans.append(("cat", cat, cat_cols))
    return ColumnTransformer(trans)

def metricas(y_true, y_pred, puntuacion, multiclase):
    """Accuracy, F1 y AUC. Multiclase: F1-macro y AUC one-vs-rest macro."""
    acc = accuracy_score(y_true, y_pred)
    if multiclase:
        return acc, f1_score(y_true, y_pred, average="macro", zero_division=0), \
               roc_auc_score(y_true, puntuacion, multi_class="ovr", average="macro")
    return acc, f1_score(y_true, y_pred, zero_division=0), roc_auc_score(y_true, puntuacion)

print("Funciones de modelos y metricas listas.")

### Paso 4 — Función `evaluar`
Para cada dataset: compara los 4 candidatos, elige el mejor **por AUC**, genera la matriz de confusión y la importancia de variables (Random Forest), y serializa el pipeline ganador con `joblib`. En validación cruzada la matriz de confusión se calcula con predicciones fuera de muestra, no sobre datos de entrenamiento.

> La importancia por impureza de Random Forest tiende a favorecer variables continuas con muchos valores distintos; se usa como orientación, no como prueba causal.

In [ ]:
def grafico_modelo(nombre, cm, clases, nombres_feat, importancias, titulo):
    """Matriz de confusion normalizada + importancia de variables (Random Forest)."""
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax[0],
                xticklabels=clases, yticklabels=clases, cbar=False)
    ax[0].set_title(f"Matriz de confusion (por fila) - {titulo}")
    ax[0].set_xlabel("Predicho"); ax[0].set_ylabel("Real")
    top = pd.Series(importancias, index=nombres_feat).sort_values().tail(10)
    top.plot(kind="barh", ax=ax[1], color="#1F5C99")
    ax[1].set_title("Top 10 variables mas importantes (Random Forest)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG, f"modelo_{nombre}.png"), dpi=120)
    plt.show()

def guardar_modelo(nombre, pipe, num_cols, cat_cols, clases, target, protocolo, fila, extra=None):
    """Serializa el pipeline completo (preprocesado + modelo) con joblib + metadata.json (Dia 11)."""
    joblib.dump(pipe, os.path.join(MOD, f"modelo_{nombre}.pkl"))
    meta = {"dataset": nombre, "target": target, "clases": clases, "modelo": fila["modelo"],
            "protocolo": protocolo, "auc": round(float(fila["auc"]), 4), "f1": round(float(fila["f1"]), 4),
            "accuracy": round(float(fila["accuracy"]), 4), "features_numericas": num_cols,
            "features_categoricas": cat_cols, "semilla": SEED, "version": "1.0",
            "fecha": EJECUCION.strftime("%Y-%m-%d %H:%M:%S")}
    meta.update(extra or {})
    with open(os.path.join(MOD, f"modelo_{nombre}.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    META.append(meta)

def evaluar(nombre, X, y, num_cols, cat_cols, clases, target):
    """Compara los 4 candidatos, elige el mejor por AUC, genera graficos y serializa el modelo.
    n < 1000 -> validacion cruzada 5x3 (un unico hold-out seria inestable); n >= 1000 -> hold-out 80/20."""
    if not permitido(nombre):
        print(f"[{nombre}] OMITIDO: uso_modelado='excluido' en vw_dim_dataset (politica acordada con el tutor).")
        return None
    multiclase = len(clases) > 2
    X = X[num_cols + cat_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    y = np.asarray(y).astype(int)
    protocolo = "CV 5x3" if len(y) < 1000 else "Hold-out 80/20"
    dist = pd.Series(y).value_counts(normalize=True).sort_index().round(3).to_dict()
    print(f"[{nombre}] n={len(y):,} | clases={len(clases)} | distribucion={dist} | protocolo={protocolo}")

    mk = lambda est: Pipeline([("prep", crear_preprocesador(num_cols, cat_cols)), ("clf", est)])
    filas, ajustados = [], {}
    if protocolo == "CV 5x3":
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
        sc = {"acc": "accuracy", "f1": "f1_macro" if multiclase else "f1",
              "auc": "roc_auc_ovr" if multiclase else "roc_auc"}
        for m, est in crear_modelos(multiclase).items():
            r = cross_validate(mk(est), X, y, cv=cv, scoring=sc, return_train_score=True)
            filas.append({"dataset": nombre, "modelo": m, "protocolo": protocolo, "n": len(y),
                          "accuracy": r["test_acc"].mean(), "f1": r["test_f1"].mean(),
                          "auc": r["test_auc"].mean(), "auc_desv": r["test_auc"].std(),
                          "brecha_acc": r["train_acc"].mean() - r["test_acc"].mean()})
    else:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
        for m, est in crear_modelos(multiclase).items():
            p = mk(est).fit(Xtr, ytr)
            prob = p.predict_proba(Xte)
            acc, f1, auc = metricas(yte, p.predict(Xte), prob if multiclase else prob[:, 1], multiclase)
            filas.append({"dataset": nombre, "modelo": m, "protocolo": protocolo, "n": len(y),
                          "accuracy": acc, "f1": f1, "auc": auc, "auc_desv": np.nan,
                          "brecha_acc": accuracy_score(ytr, p.predict(Xtr)) - acc})
            ajustados[m] = p
    res = pd.DataFrame(filas)
    cand = res[res.modelo != BASE_NAME].sort_values(["auc", "f1"], ascending=False)
    mejor = cand.iloc[0]

    # Modelo final, matriz de confusion honesta e importancia de variables
    if protocolo == "CV 5x3":
        final = mk(crear_modelos(multiclase)[mejor.modelo]).fit(X, y)
        pred = cross_val_predict(mk(crear_modelos(multiclase)[mejor.modelo]), X, y,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
        rf = mk(crear_modelos(multiclase)["Random Forest"]).fit(X, y)
        y_ref = y
    else:
        final, pred, y_ref = ajustados[mejor.modelo], ajustados[mejor.modelo].predict(Xte), yte
        rf = ajustados["Random Forest"]
    nombres = [n.split("__", 1)[1] for n in rf.named_steps["prep"].get_feature_names_out()]
    imp_ord = pd.Series(rf.named_steps["clf"].feature_importances_, index=nombres).sort_values(ascending=False)
    print("Variables mas influyentes (Random Forest):", ", ".join(f"{k} ({v:.2f})" for k, v in imp_ord.head(4).items()))
    grafico_modelo(nombre, confusion_matrix(y_ref, pred), clases, nombres,
                   rf.named_steps["clf"].feature_importances_, f"{nombre} ({mejor.modelo})")
    guardar_modelo(nombre, final, num_cols, cat_cols, clases, target, protocolo, mejor, extra={
        "n": int(len(y)), "distribucion_clases": {clases[k]: v for k, v in dist.items()},
        "variables_influyentes": [f"{k} ({v:.2f})" for k, v in imp_ord.head(4).items()]})

    RESULTADOS.append(res.assign(target=target))
    cols = ["modelo", "accuracy", "f1", "auc"] + (["auc_desv"] if protocolo == "CV 5x3" else []) + ["brecha_acc"]
    display(res[cols].round(3).reset_index(drop=True))
    print(f"-> Mejor modelo por AUC: {mejor.modelo} (AUC={mejor.auc:.3f}, F1={mejor.f1:.3f}) | "
          f"baseline Accuracy={res[res.modelo == BASE_NAME].accuracy.iloc[0]:.3f}")
    return res

print("Funcion evaluar() lista.")

---
## Modelos por dataset

### B1_ACADSTRESS — estrés académico (n = 140, 5 clases)
**Target:** índice de estrés 1-5. **Feature Engineering:** *Label Encoding* de la etapa académica (orden natural) · *One-Hot* de entorno, estrategia de afrontamiento y hábitos · variable derivada `presion_total = pares + hogar + competencia`.

In [ ]:
# B1_ACADSTRESS - estres academico (n=140). Target: indice 1-5 (multiclase)
d = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student Academic Stress Real World Dataset",
                "academic Stress level - maintainance 1.csv"))
d.columns = [c.strip() for c in d.columns]
T = "Rate your academic stress index"
d = d[d[T].notna()].copy()
c_comp = [c for c in d.columns if "competition" in c][0]
c_cop  = [c for c in d.columns if "coping" in c][0]
c_hab  = [c for c in d.columns if "bad habits" in c][0]

X = pd.DataFrame({
    "etapa": d["Your Academic Stage"].map({"high school": 0, "undergraduate": 1, "post-graduate": 2}),  # Label Encoding (orden natural)
    "presion_pares": d["Peer pressure"], "presion_hogar": d["Academic pressure from your home"],
    "competencia": d[c_comp],
    "entorno": d["Study Environment"], "afrontamiento": d[c_cop].str[:25], "habitos": d[c_hab],      # One-Hot (sin orden)
})
X["presion_total"] = X["presion_pares"] + X["presion_hogar"] + X["competencia"]                       # variable derivada
y = d[T].astype(int) - 1
res_acad = evaluar("B1_ACADSTRESS", X, y,
    num_cols=["etapa", "presion_pares", "presion_hogar", "competencia", "presion_total"],
    cat_cols=["entorno", "afrontamiento", "habitos"],
    clases=["1", "2", "3", "4", "5"], target="Indice de estres academico (1-5)")

**Opinión.** El mejor modelo (Random Forest) alcanza un AUC ≈ 0.73, pero su **Accuracy (≈ 0.40) no supera al baseline (0.40)**: ordena los niveles de estrés mejor que el azar, pero no acierta el nivel exacto. La brecha train-test ≈ 0.43 revela sobreajuste fuerte. Con 140 filas y clases de apenas 6 y 9 casos, predecir 5 niveles es demasiado ambicioso: el resultado es **exploratorio**. La variable derivada `presion_total` encabeza la importancia (≈ 0.25), lo que respalda el hallazgo del Objetivo 1 sobre la presión académica. Mejora posible: trabajar con los 3 niveles de la capa armonizada o ampliar la muestra.

### B1_ANXIETY — ansiedad GAD-7 (estudiantes, n = 9.302)
**Target:** GAD-7 ≥ 10 (ansiedad moderada o severa; solo ≈ 17 % de los casos). **Fuga evitada:** los ítems GAD1-7 y GADE se excluyen porque `GAD_T` es su suma. **Feature Engineering:** variables derivadas con cortes clínicos (`swl_insatisfecho` si SWL ≤ 14; `spin_alto` si SPIN ≥ 19) y `horas_juego` acotada a 112 h/semana (regla de calidad R4).

In [ ]:
# B1_ANXIETY - ansiedad GAD-7 (estudiantes). Target: GAD-7 >= 10 (ansiedad moderada o severa)
d = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student anxiety dataset", "anxiety.csv"), encoding="latin1")
d = d[d["Work"].isin(["Student at college / university", "Student at school"]) & d["GAD_T"].notna()].copy()

# Los items GAD1-7 y GADE se EXCLUYEN: GAD_T es su suma y filtrarian el target (fuga de datos)
X = pd.DataFrame({
    "swl": d["SWL_T"], "spin": d["SPIN_T"],
    "horas_juego": pd.to_numeric(d["Hours"], errors="coerce").clip(0, 112),   # regla de calidad R4: max 112 h/semana
    "edad": pd.to_numeric(d["Age"], errors="coerce").where(lambda s: s.between(13, 90)),
    "narcisismo": pd.to_numeric(d["Narcissism"], errors="coerce"),
    "genero": d["Gender"], "grado": d["Degree"], "ocupacion": d["Work"], "plataforma": d["Platform"],
})
X["swl_insatisfecho"] = (X["swl"] <= 14).astype(int)    # corte clinico SWL: <=14 insatisfaccion con la vida
X["spin_alto"] = (X["spin"] >= 19).astype(int)          # corte de ansiedad social (SPIN >= 19)
y = (d["GAD_T"] >= 10).astype(int)
res_anx = evaluar("B1_ANXIETY", X, y,
    num_cols=["swl", "spin", "horas_juego", "edad", "narcisismo", "swl_insatisfecho", "spin_alto"],
    cat_cols=["genero", "grado", "ocupacion", "plataforma"],
    clases=["No alto (<10)", "Alto (>=10)"], target="GAD-7 >= 10")

**Opinión — el ejemplo de la trampa del Accuracy (Día 5).** El baseline "nunca hay ansiedad alta" obtiene **83 % de Accuracy con F1 = 0**: supera en Accuracy a todos los modelos, pero no detecta ni un solo caso. La Regresión Logística (Accuracy ≈ 0.73) tiene **F1 ≈ 0.45 y AUC ≈ 0.77**, el mejor equilibrio, y apenas sobreajusta (brecha ≈ 0.00 frente a ≈ 0.15 del Random Forest). Aquí gana el modelo más simple. SPIN (ansiedad social) y SWL (satisfacción vital) dominan la importancia, coherente con las correlaciones del Objetivo 1. Limitación: la población original son jugadores online.

### B1_MHSURVEY — salud mental universitaria (n = 87, 3 clases)
**Target:** depresión en 3 niveles (Bajo 1-2 / Moderado 3 / Alto 4-5, como en la capa armonizada). **Fuga evitada:** `anxiety` se excluye (r = 0.84 con el target). **Feature Engineering:** `cgpa` y `average_sleep` pasan de rangos de texto a orden numérico (*Label Encoding* ordinal) · `deporte` ordinal · One-Hot de residencia y año · derivadas `carga_presion` (interacción) y `bienestar = satisfacción − aislamiento`.

In [ ]:
# B1_MHSURVEY - salud mental universitaria (n=87). Target: depresion en 3 niveles (Bajo 1-2 / Moderado 3 / Alto 4-5)
d = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student Mental Health Survey", "MentalHealthSurvey.csv"))
d.columns = [c.strip() for c in d.columns]

def orden_rango(s):
    """'2.0-2.5' -> 2.0 ; '2-4 hrs' -> 2 : el limite inferior del rango sirve como orden (Label Encoding ordinal)."""
    return pd.to_numeric(s.astype(str).str.extract(r"(\d+\.?\d*)")[0], errors="coerce")

# 'anxiety' se excluye: es el otro resultado de la encuesta (r=0.84 con el target) y haria trivial la tarea
X = pd.DataFrame({
    "edad": d["age"], "cgpa": orden_rango(d["cgpa"]), "sueno": orden_rango(d["average_sleep"]),
    "deporte": d["sports_engagement"].map({"No Sports": 0, "1-3 times": 1, "4-6 times": 2, "7+ times": 3}),
    "satisfaccion": d["study_satisfaction"], "carga": d["academic_workload"], "presion": d["academic_pressure"],
    "financiero": d["financial_concerns"], "relaciones": d["social_relationships"],
    "aislamiento": d["isolation"], "inseguridad": d["future_insecurity"],
    "genero": d["gender"], "nivel": d["degree_level"], "residencia": d["residential_status"],
    "discriminacion": d["campus_discrimination"], "anio": d["academic_year"],
})
X["carga_presion"] = X["carga"] * X["presion"]               # variable de interaccion
X["bienestar"] = X["satisfaccion"] - X["aislamiento"]
y = pd.cut(d["depression"], [0, 2, 3, 5], labels=[0, 1, 2]).astype(int)
res_mhs = evaluar("B1_MHSURVEY", X, y,
    num_cols=["edad", "cgpa", "sueno", "deporte", "satisfaccion", "carga", "presion", "financiero",
              "relaciones", "aislamiento", "inseguridad", "carga_presion", "bienestar"],
    cat_cols=["genero", "nivel", "residencia", "discriminacion", "anio"],
    clases=["Bajo", "Moderado", "Alto"], target="Depresion autoreportada (3 niveles)")

**Opinión.** AUC ≈ 0.76-0.79 y Accuracy ≈ 0.62 frente a 0.45 del baseline: señal real. Sin embargo, la desviación entre repeticiones (±0.07) es del tamaño de las diferencias entre modelos, así que **XGBoost, Regresión Logística y Random Forest son estadísticamente indistinguibles con n = 87**; el sobreajuste de los árboles es alto (brecha ≈ 0.38). Lo más relevante: la variable derivada `bienestar` queda segunda en importancia, detrás de `aislamiento`, confirmando que el aislamiento social es el predictor central, igual que en el Objetivo 1.

### B2_SMMH — redes sociales y salud mental (n = 481)
**Target:** "se siente deprimido/a" ≥ 4 sobre 5. Se excluye la pregunta 19 (interés en actividades) por ser síntoma del mismo constructo. **Feature Engineering:** *One-Hot* de las plataformas (multi-selección separada en dummies) · `tiempo_diario` ordinal · derivadas `uso_ansioso = (inquietud + distracción fácil + preocupación) / 3` e `impacto_social = (comparación + búsqueda de validación) / 2`.

In [ ]:
# B2_SMMH - redes sociales y salud mental (n=481). Target: "se siente deprimido/a" >= 4 sobre 5
d = pd.read_csv(os.path.join(BASE, "Bloque 2", "2 Social Media and Mental Health", "smmh.csv"))
c = list(d.columns)
num = lambda i: pd.to_numeric(d[c[i]], errors="coerce")
d = d[num(18).notna()].copy()                                  # target (pregunta 18)
num = lambda i: pd.to_numeric(d[c[i]], errors="coerce")

X = pd.DataFrame({
    "edad": num(1),
    "tiempo_diario": d[c[8]].map({"Less than an Hour": 0, "Between 1 and 2 hours": 1, "Between 2 and 3 hours": 2,
                                  "Between 3 and 4 hours": 3, "Between 4 and 5 hours": 4, "More than 5 hours": 5}),
    "uso_sin_proposito": num(9), "distraccion_redes": num(10), "inquietud_sin_redes": num(11),
    "distraccion_facil": num(12), "preocupacion": num(13), "dificultad_concentracion": num(14),
    "comparacion_social": num(15), "sentimiento_comparacion": num(16), "busqueda_validacion": num(17),
    "problemas_sueno": num(20),
    "genero": d[c[2]], "relacion": d[c[3]], "ocupacion": d[c[4]],
})
# La pregunta 19 (interes en actividades) se excluye: es un sintoma del mismo constructo que el target.
X["uso_ansioso"] = (X["inquietud_sin_redes"] + X["distraccion_facil"] + X["preocupacion"]) / 3   # interaccion
X["impacto_social"] = (X["comparacion_social"] + X["busqueda_validacion"]) / 2
# One-Hot de la multi-seleccion de plataformas (solo las que aparecen en >= 20 respuestas)
plat = d[c[7]].fillna("").str.split(r",\s*")
conteo = pd.Series([p for l in plat for p in l if p]).value_counts()
cols_plat = []
for p in [p for p, k in conteo.items() if k >= 20]:
    col = "usa_" + re.sub(r"\W+", "_", p.lower())
    X[col] = plat.apply(lambda l: int(p in l)); cols_plat.append(col)
y = (num(18) >= 4).astype(int)
res_smmh = evaluar("B2_SMMH", X, y,
    num_cols=["edad", "tiempo_diario", "uso_sin_proposito", "distraccion_redes", "inquietud_sin_redes",
              "distraccion_facil", "preocupacion", "dificultad_concentracion", "comparacion_social",
              "sentimiento_comparacion", "busqueda_validacion", "problemas_sueno",
              "uso_ansioso", "impacto_social"] + cols_plat,
    cat_cols=["genero", "relacion", "ocupacion"],
    clases=["Bajo (<4)", "Alto (>=4)"], target="Se siente deprimido/a >= 4")

**Opinión.** Es el dataset de hábitos digitales con señal real: AUC ≈ 0.79 y Accuracy ≈ 0.74 frente a 0.54 del baseline; Regresión Logística y Random Forest empatan, así que el patrón es esencialmente lineal. **Dos de las cuatro variables más importantes son derivadas** (`uso_ansioso`, `impacto_social`), y ni el tiempo diario ni las plataformas figuran entre las primeras: confirma el hallazgo del Objetivo 1 (importa el *patrón de uso* —preocupación, concentración, comparación—, no cuánto tiempo se usa). Nota: `uso_ansioso` (r ≈ 0.56) no supera a su mejor componente, `preocupación` (r ≈ 0.59); aporta estabilidad al promediar tres señales.

### B3_DEPSTUDENT — depresión estudiantil (n = 502)
**Target:** Depression (Yes/No). **Feature Engineering:** *Label Encoding* ordinal de sueño y dieta · variable binaria de pensamientos suicidas y antecedentes · derivada `riesgo_alto` (presión ≥ 4 **y** estrés financiero ≥ 4).

In [ ]:
# B3_DEPSTUDENT - depresion estudiantil (n=502). Target: Depression Yes/No
d = pd.read_csv(os.path.join(BASE, "Bloque 3", "3 Student_Mental_Health_Analysis GITHUB",
                "Depression+Student+Dataset.csv"))
X = pd.DataFrame({
    "genero": d["Gender"], "edad": d["Age"], "presion": d["Academic Pressure"],
    "satisfaccion": d["Study Satisfaction"],
    "sueno": d["Sleep Duration"].map({"Less than 5 hours": 0, "5-6 hours": 1, "7-8 hours": 2, "More than 8 hours": 3}),
    "dieta": d["Dietary Habits"].map({"Unhealthy": 0, "Moderate": 1, "Healthy": 2}),     # Label Encoding ordinal
    "pens_suicidas": (d["Have you ever had suicidal thoughts ?"] == "Yes").astype(int),
    "horas_estudio": d["Study Hours"], "financiero": pd.to_numeric(d["Financial Stress"], errors="coerce"),
    "antecedentes": (d["Family History of Mental Illness"] == "Yes").astype(int),
})
X["riesgo_alto"] = ((X["presion"] >= 4) & (X["financiero"] >= 4)).astype(int)            # interaccion
y = (d["Depression"] == "Yes").astype(int)
res_dep = evaluar("B3_DEPSTUDENT", X, y,
    num_cols=["edad", "presion", "satisfaccion", "sueno", "dieta", "pens_suicidas", "horas_estudio",
              "financiero", "antecedentes", "riesgo_alto"],
    cat_cols=["genero"], clases=["Sin depresion", "Con depresion"], target="Depression (Yes/No)")

### ⚠️ Alerta: un resultado demasiado perfecto
La Regresión Logística obtiene **AUC ≈ 0.998 y Accuracy ≈ 0.98**. Ningún dataset real de salud mental alcanza esa separabilidad con diez variables gruesas. El diagnóstico (celda siguiente) descarta una fuga por columna —ninguna variable supera AUC ≈ 0.77 por separado—, pero muestra que **una combinación lineal de las variables reproduce la etiqueta casi sin error**, patrón típico de etiquetas generadas por una fórmula. No se puede afirmar que sea sintético, pero **es un indicio serio**. Dado que la política acordada con el tutor es no modelar datos sintéticos, este resultado **se reporta con cautela y no se usa como evidencia sustantiva**; conviene consultarlo con el tutor. Este dataset sustenta además el KPI de riesgo crítico (72.7 %).

**Si el tutor lo confirma**, basta ejecutar `UPDATE dim_dataset SET uso_modelado='excluido', calidad='tecnico' WHERE codigo='B3_DEPSTUDENT';` para que la exclusión se propague a la estrella, a Power BI y a este notebook (que lo omitirá automáticamente).

In [ ]:
# DIAGNOSTICO: el AUC ~ 1 es demasiado perfecto para datos clinicos reales. ?Hay fuga de datos o una formula detras?
if permitido("B3_DEPSTUDENT"):
    from sklearn.tree import DecisionTreeClassifier
    Xd = X[["edad", "presion", "satisfaccion", "sueno", "dieta", "pens_suicidas", "horas_estudio",
            "financiero", "antecedentes"]].copy()
    Xd = Xd.fillna(Xd.median())
    aucs = {c: max(roc_auc_score(y, Xd[c]), 1 - roc_auc_score(y, Xd[c])) for c in Xd}
    print("AUC de cada variable POR SEPARADO (ninguna deberia acercarse a 1 si no hay fuga):")
    print(pd.Series(aucs).sort_values(ascending=False).round(3).to_string())
    auc_arbol = {}
    for k in (2, 4):
        arbol = DecisionTreeClassifier(max_depth=k, random_state=SEED)
        auc_arbol[k] = cross_validate(arbol, Xd, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                                      scoring="roc_auc")["test_score"].mean()
        print(f"Arbol de profundidad {k} (reglas simples): AUC = {auc_arbol[k]:.3f}")
    diag = {"auc_por_variable": {k: round(float(v), 3) for k, v in aucs.items()},
            "auc_arbol_prof2": round(float(auc_arbol[2]), 3), "auc_arbol_prof4": round(float(auc_arbol[4]), 3)}
    for m_ in META:
        if m_["dataset"] == "B3_DEPSTUDENT":
            m_["diagnostico"] = diag
            with open(os.path.join(MOD, "modelo_B3_DEPSTUDENT.json"), "w", encoding="utf-8") as f:
                json.dump(m_, f, indent=2, ensure_ascii=False)
    print("\nConclusion: ninguna columna filtra la etiqueta (maximo ~0.77 por separado), pero una combinacion lineal de"
          "\nvarias variables la reproduce casi sin error. Es un patron tipico de etiquetas generadas por una formula,"
          "\nno de datos clinicos reales: tratar este resultado con cautela y consultarlo con el tutor.")
else:
    print("B3_DEPSTUDENT no se modela (excluido en la estrella): sin diagnostico.")

### B4_DASS — DASS-42, 18-25 años (n = 23.039, 3 clases)
**Target:** nivel de estrés por baremos (Bajo ≤ 28 / Moderado 29-39 / Alto ≥ 40). **Predictores:** perfil demográfico + personalidad TIPI. **No se usa ningún ítem DASS**: los de estrés definen el target. **Feature Engineering:** las 10 respuestas TIPI se convierten en las 5 puntuaciones Big Five (invirtiendo los ítems inversos con `8 − x`) · *One-Hot* de género, entorno de infancia, lengua nativa y estado civil.

In [ ]:
# B4_DASS - DASS-42 (18-25 anios). Target: nivel de estres por baremos (Bajo <=28 / Moderado 29-39 / Alto >=40)
d = pd.read_csv(os.path.join(BASE, "Bloque 4", "4 Depression Anxiety Stress Scales Responses", "data.csv"),
                sep="\t", low_memory=False)
d = d[(d["age"] >= 18) & (d["age"] <= 25)].copy()
str_items = [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]
puntaje = d[[f"Q{i}A" for i in str_items]].sum(axis=1)
y = pd.cut(puntaje, [0, 28, 39, 100], labels=[0, 1, 2]).astype(int)

# Predictores: perfil demografico + personalidad (TIPI). NINGUN item DASS: los de estres definen el target
# y los de depresion/ansiedad lo harian casi trivial. Religion, orientacion y raza se excluyen por etica.
t = d[[f"TIPI{i}" for i in range(1, 11)]].replace(0, np.nan)          # 0 = sin respuesta
X = pd.DataFrame({
    "extraversion": (t.TIPI1 + (8 - t.TIPI6)) / 2,                    # puntuaciones Big Five (items inversos 8 - x)
    "amabilidad": ((8 - t.TIPI2) + t.TIPI7) / 2,
    "responsabilidad": (t.TIPI3 + (8 - t.TIPI8)) / 2,
    "estab_emocional": ((8 - t.TIPI4) + t.TIPI9) / 2,
    "apertura": (t.TIPI5 + (8 - t.TIPI10)) / 2,
    "edad": d["age"], "educacion": d["education"].replace(0, np.nan),
    "familia": d["familysize"].clip(0, 10),
    "genero": d["gender"].map({1: "M", 2: "F", 3: "O"}),
    "infancia": d["urban"].map({1: "rural", 2: "suburbano", 3: "urbano"}),
    "ingles_nativo": d["engnat"].map({1: "si", 2: "no"}),
    "estado_civil": d["married"].map({1: "soltero", 2: "casado", 3: "previo"}),
})
res_dass = evaluar("B4_DASS", X, y,
    num_cols=["extraversion", "amabilidad", "responsabilidad", "estab_emocional", "apertura",
              "edad", "educacion", "familia"],
    cat_cols=["genero", "infancia", "ingles_nativo", "estado_civil"],
    clases=["Bajo", "Moderado", "Alto"], target="Nivel de estres DASS-42 (baremos)")

**Opinión.** Predecir el estrés solo a partir del perfil y la personalidad da **AUC ≈ 0.76 y Accuracy ≈ 0.58 frente a 0.36 del baseline**. La **estabilidad emocional domina con diferencia** (importancia ≈ 0.31, el triple que cualquier otro rasgo), coherente con la relación psicológica entre neuroticismo y estrés. Regresión Logística y XGBoost empatan, y XGBoost apenas sobreajusta (brecha ≈ 0.03) frente al Random Forest (≈ 0.38): la regularización importa. Con ≈ 0.58 de Accuracy en 3 clases, la personalidad explica una parte del estrés, no su totalidad.

### B4_MENTALHEALTH — búsqueda de tratamiento (estudiantes, n = 54.207)
**Target:** ha buscado tratamiento (Yes/No) — es una **conducta**, no un nivel de estrés. **Feature Engineering:** *Label Encoding* ordinal de `dias_en_casa`, `estres_creciente` y `cambios_humor` · *One-Hot* del resto · derivada `vulnerabilidad = antecedentes + historial propio`.

In [ ]:
# B4_MENTALHEALTH - encuesta masiva (estudiantes). Target: ha buscado tratamiento (Yes/No)
d = pd.read_csv(os.path.join(BASE, "Bloque 4", "4 Mental Health Dataset", "Mental Health dataset1.csv"))
d = d[d["Occupation"] == "Student"].copy()
X = pd.DataFrame({
    "genero": d["Gender"], "autonomo": d["SelfEmployed"],
    "antecedentes": (d["FamilyHistory"] == "Yes").astype(int),
    "dias_en_casa": d["DaysIndoors"].map({"Go out Every day": 0, "1-14 days": 1, "15-30 days": 2,
                                          "31-60 days": 3, "More than 2 months": 4}),     # Label Encoding ordinal
    "estres_creciente": d["IncreasingStress"].map({"No": 0, "Maybe": 1, "Yes": 2}),
    "cambios_humor": d["MoodSwings"].map({"Low": 0, "Medium": 1, "High": 2}),
    "cambio_habitos": d["HabitsChange"], "historial_sm": d["MentalHealthHistory"],
    "debilidad_social": d["SocialWeakness"], "afrontamiento": d["CopingStruggles"],
    "interes_trabajo": d["WorkInterest"], "entrevista": d["MentalHealthInterview"],
    "conoce_opciones": d["CareOptions"],
})
X["vulnerabilidad"] = X["antecedentes"] + (X["historial_sm"] == "Yes").astype(int)         # variable derivada
y = (d["Treatment"] == "Yes").astype(int)
res_mh = evaluar("B4_MENTALHEALTH", X, y,
    num_cols=["antecedentes", "dias_en_casa", "estres_creciente", "cambios_humor", "vulnerabilidad"],
    cat_cols=["genero", "autonomo", "cambio_habitos", "historial_sm", "debilidad_social", "afrontamiento",
              "interes_trabajo", "entrevista", "conoce_opciones"],
    clases=["Sin tratamiento", "Con tratamiento"], target="Ha buscado tratamiento (Yes/No)")

**Opinión.** XGBoost: AUC ≈ 0.77 y Accuracy ≈ 0.72 frente a 0.51, sin sobreajuste (brecha ≈ 0.00). Los **antecedentes familiares encabezan la importancia (≈ 0.27) y `estres_creciente` no figura entre las primeras**: el modelo reproduce el hallazgo del Objetivo 1 (el estrés percibido no mueve la búsqueda de ayuda; la familiaridad con la salud mental sí). Cautela: `conoce_opciones` (≈ 0.16) puede ser *consecuencia* de estar en tratamiento y no su causa (causalidad inversa).

---
## NLP — B3_DREADDIT (detección de estrés en texto)
Se comparan **tres enfoques** sobre la partición train/test original: (1) las variables psicolingüísticas precalculadas (LIWC, DAL, legibilidad, sentimiento), (2) **TF-IDF** de unigramas y bigramas con tres clasificadores, y (3) un **híbrido** que combina ambas. No se eliminan las *stop-words*: pronombres como "I" o "me" son justamente señal de estrés. Se excluyen `confidence` (no existe al predecir), `subreddit` (casi la etiqueta) y `social_*`. Se aplican las reglas de calidad R6 (alinear columnas por nombre) y R7 (excluir textos `#NAME?`); no hay solapamiento de `post_id` entre train y test, así que no hay fuga.

In [ ]:
# B3_DREADDIT - NLP: ?el texto permite detectar estres? Particion train/test ORIGINAL de los autores
assert permitido("B3_DREADDIT"), "B3_DREADDIT figura como excluido en la estrella: no se modela."
ruta = os.path.join(BASE, "Bloque 3", "3 Stress Analysis in Social Media Dataset")
tr = pd.read_csv(os.path.join(ruta, "dreaddit-train.csv"))
te = pd.read_csv(os.path.join(ruta, "dreaddit-test.csv"))
corruptos = ["#NAME?", "#REF!", "#VALUE!"]                     # regla de calidad R7
tr = tr[~tr["text"].str.strip().isin(corruptos)].copy()
te = te[~te["text"].str.strip().isin(corruptos)].copy()
te = te[tr.columns]                                            # regla R6: alinear columnas por nombre
print(f"train={len(tr):,} | test={len(te):,} | solapamiento de post_id: {len(set(tr.post_id) & set(te.post_id))}")

# Variables psicolinguisticas precalculadas (LIWC, DAL, legibilidad, sentimiento).
# Se excluyen: 'confidence' (acuerdo de anotadores, no existe al predecir), 'subreddit' (casi etiqueta) y 'social_*'.
LEX = [c for c in tr.columns if c.startswith(("lex_", "syntax_"))] + ["sentiment"]
print("Variables psicolinguisticas:", len(LEX))

tf = lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True)   # sin stop-words: 'I', 'me' son senal
def lr(C=1.0): return LogisticRegression(max_iter=3000, C=C, class_weight="balanced")
boost = (XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                       random_state=SEED, n_jobs=-1, eval_metric="logloss") if BOOST_NAME == "XGBoost"
         else HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=SEED))
pipes = {
    BASE_NAME: DummyClassifier(strategy="most_frequent"),
    "LIWC + Regresion Logistica": Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), LEX)])), ("clf", lr(0.1))]),
    "LIWC + Random Forest": Pipeline([("prep", ColumnTransformer([("num", "passthrough", LEX)])),
                                      ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                       class_weight="balanced", random_state=SEED, n_jobs=-1))]),
    f"LIWC + {BOOST_NAME}": Pipeline([("prep", ColumnTransformer([("num", "passthrough", LEX)])), ("clf", boost)]),
    "TF-IDF + Regresion Logistica": Pipeline([("prep", ColumnTransformer([("tf", tf(), "text")])), ("clf", lr(1.0))]),
    "TF-IDF + SVM lineal": Pipeline([("prep", ColumnTransformer([("tf", tf(), "text")])),
                                     ("clf", LinearSVC(C=0.5, class_weight="balanced"))]),
    "TF-IDF + Naive Bayes": Pipeline([("prep", ColumnTransformer([("tf", tf(), "text")])), ("clf", MultinomialNB(alpha=0.5))]),
    "Hibrido (TF-IDF + LIWC) + RegLog": Pipeline([("prep", ColumnTransformer([("tf", tf(), "text"), ("num", StandardScaler(), LEX)])),
                                                 ("clf", lr(0.5))]),
}
ytr, yte = tr["label"].astype(int).values, te["label"].astype(int).values
filas, ajust = [], {}
for nombre_m, p in pipes.items():
    p.fit(tr, ytr)
    punt = p.predict_proba(te)[:, 1] if hasattr(p, "predict_proba") else p.decision_function(te)
    acc, f1, auc = metricas(yte, p.predict(te), punt, False)
    filas.append({"dataset": "B3_DREADDIT", "modelo": nombre_m, "protocolo": "Train/Test original", "n": len(tr) + len(te),
                  "accuracy": acc, "f1": f1, "auc": auc, "auc_desv": np.nan,
                  "brecha_acc": accuracy_score(ytr, p.predict(tr)) - acc})
    ajust[nombre_m] = p
res_nlp = pd.DataFrame(filas)
display(res_nlp[["modelo", "accuracy", "f1", "auc", "brecha_acc"]].round(3).reset_index(drop=True))

mejor = res_nlp[res_nlp.modelo != BASE_NAME].sort_values(["auc", "f1"], ascending=False).iloc[0]
pm = ajust[mejor.modelo]
cm = confusion_matrix(yte, pm.predict(te))

# Interpretabilidad: palabras que mas empujan hacia 'estres' en la regresion logistica sobre TF-IDF
pl = ajust["TF-IDF + Regresion Logistica"]
vocab = np.array(pl.named_steps["prep"].named_transformers_["tf"].get_feature_names_out())
coef = pl.named_steps["clf"].coef_[0]
orden = np.argsort(coef)
print("Terminos que empujan hacia ESTRES:", ", ".join(vocab[orden[-10:]][::-1]))
print("Terminos que empujan hacia NO ESTRES:", ", ".join(vocab[orden[:10]]))
terminos = pd.Series(np.r_[coef[orden[:12]], coef[orden[-12:]]], index=np.r_[vocab[orden[:12]], vocab[orden[-12:]]])

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax[0],
            xticklabels=["No estres", "Estres"], yticklabels=["No estres", "Estres"], cbar=False)
ax[0].set_title(f"Matriz de confusion (por fila)\n{mejor.modelo}"); ax[0].set_xlabel("Predicho"); ax[0].set_ylabel("Real")
terminos.plot(kind="barh", ax=ax[1], color=["#55A868" if v < 0 else "#C44E52" for v in terminos.values])
ax[1].set_title("Terminos que empujan a 'No estres' (verde) / 'Estres' (rojo)")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "modelo_B3_DREADDIT.png"), dpi=120); plt.show()

joblib.dump(pm, os.path.join(MOD, "modelo_B3_DREADDIT.pkl"))
# El mejor modelo usa variables LIWC (diccionario con licencia) que NO se pueden calcular sobre texto nuevo.
# Por eso se guarda tambien una version desplegable que solo necesita el texto:
joblib.dump(ajust["TF-IDF + Regresion Logistica"], os.path.join(MOD, "modelo_B3_DREADDIT_solo_texto.pkl"))
meta = {"dataset": "B3_DREADDIT", "target": "label estres (0/1)", "clases": ["No estres", "Estres"],
        "modelo": mejor.modelo, "protocolo": "Train/Test original", "auc": round(float(mejor.auc), 4),
        "f1": round(float(mejor.f1), 4), "accuracy": round(float(mejor.accuracy), 4),
        "features_numericas": LEX, "features_texto": ["text"], "semilla": SEED, "version": "1.0",
        "fecha": EJECUCION.strftime("%Y-%m-%d %H:%M:%S")}
meta.update({"n": len(tr) + len(te), "n_train": len(tr), "n_test": len(te),
             "distribucion_clases": {"No estres": round(float((yte == 0).mean()), 3), "Estres": round(float((yte == 1).mean()), 3)},
             "terminos_estres": [str(t) for t in vocab[orden[-10:]][::-1]],
             "terminos_no_estres": [str(t) for t in vocab[orden[:10]]],
             "recall_estres": round(float(cm[1, 1] / cm[1].sum()), 3), "especificidad": round(float(cm[0, 0] / cm[0].sum()), 3)})
with open(os.path.join(MOD, "modelo_B3_DREADDIT.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
META.append(meta); RESULTADOS.append(res_nlp.assign(target="label estres (0/1)"))
print(f"-> Mejor: {mejor.modelo} (AUC={mejor.auc:.3f}, F1={mejor.f1:.3f})")

**Opinión.** Los tres enfoques rinden de forma muy parecida (**AUC ≈ 0.82-0.84, Accuracy ≈ 0.72-0.75 frente a 0.52 del baseline**); el híbrido es el mejor (AUC ≈ 0.84), pero con 715 textos de prueba las diferencias de ≈ 0.01 no son concluyentes: **una Regresión Logística sobre LIWC (AUC ≈ 0.835) iguala a XGBoost**. Los modelos sobre TF-IDF sobreajustan más (brecha ≈ 0.22-0.27) que el híbrido (≈ 0.08) y la LR sobre LIWC (≈ 0.04). El modelo detecta ≈ 78 % de los textos con estrés (recall) y descarta correctamente ≈ 71 % de los que no lo tienen.

**Qué dice el lenguaje.** Los términos que más empujan hacia *estrés* son `me`, `my`, `feel`, `am`, `anxiety`, `don know`: **lenguaje centrado en uno mismo**. Hacia *no estrés*: `you`, `we`, `her`, `she`, `your`: **lenguaje dirigido a otras personas**. Coincide con el hallazgo del Objetivo 1 (la primera persona aparece 10.8 % en posts de estrés frente a 7.1 %) y es un marcador clásico de rumiación.

**Despliegue.** El mejor modelo necesita variables LIWC (diccionario con licencia) que no se pueden calcular sobre texto nuevo. Por eso se guarda también `modelo_B3_DREADDIT_solo_texto.pkl` (TF-IDF + Regresión Logística, AUC ≈ 0.82), que solo requiere el texto. **Limitación:** el corpus procede de subreddits de PTSD, relaciones y violencia doméstica; el estrés capturado es relacional/clínico, no estrictamente académico.

### Uso del modelo serializado (Día 11)
Ejemplo de cómo se reutiliza un modelo guardado: se carga el `.pkl` y se puntúa texto nuevo sin reentrenar. Es la base para una eventual API con FastAPI, tal como se vio en la sesión de industrialización.

In [ ]:
# Ejemplo de uso del modelo serializado (Dia 11): cargar el .pkl y puntuar texto nuevo, sin reentrenar
modelo_txt = joblib.load(os.path.join(MOD, "modelo_B3_DREADDIT_solo_texto.pkl"))
ejemplos = pd.DataFrame({"text": [
    "I can't sleep and I feel like everything is falling apart, I don't know what to do anymore.",
    "We went hiking with her family last weekend and you should see the photos, it was great.",
]})
prob = modelo_txt.predict_proba(ejemplos)[:, 1]
for t, p_ in zip(ejemplos["text"], prob):
    print(f"P(estres) = {p_:.2f} | {t[:70]}...")
print("\nAviso: herramienta de investigacion, NO de diagnostico clinico.")

---
## Resumen comparativo y exportación
Tabla con el mejor modelo de cada dataset, comparado con su baseline, y gráfico de AUC. Se exportan los resultados a CSV (listos para Power BI en la Entrega 3) y un `manifest_modelos.json` con la versión, fecha y métricas de cada modelo.

In [ ]:
RES = pd.concat(RESULTADOS, ignore_index=True)
mejores = (RES[RES.modelo != BASE_NAME].sort_values(["dataset", "auc"], ascending=[True, False])
           .groupby("dataset").head(1))
base = RES[RES.modelo == BASE_NAME][["dataset", "accuracy"]].rename(columns={"accuracy": "acc_baseline"})
RESUMEN = mejores.merge(base, on="dataset")
RESUMEN["mejora_vs_baseline"] = RESUMEN["accuracy"] - RESUMEN["acc_baseline"]

LECTURA = {   # como se lee cada resultado en la capa armonizada (dim_nivel_estres / dim_mapeo_estres)
    "B1_ACADSTRESS": "Indice 1-5: 1-2 Bajo, 3 Moderado, 4-5 Alto",
    "B1_ANXIETY": "GAD-7 >= 10 equivale a nivel Alto",
    "B1_MHSURVEY": "Depresion 1-5 mapeada a Bajo / Moderado / Alto",
    "B2_SMMH": "Valor >= 4 equivale a nivel Alto",
    "B3_DREADDIT": "Etiqueta 0 = Bajo, 1 = Alto",
    "B3_DEPSTUDENT": "No = Bajo, Yes = Alto",
    "B4_DASS": "Baremos DASS: <=28 Bajo, 29-39 Moderado, >=40 Alto",
    "B4_MENTALHEALTH": "Conducta de busqueda de ayuda (no es nivel de estres): alimenta el KPI de tratamiento",
}
RESUMEN["lectura_capa_armonizada"] = RESUMEN["dataset"].map(LECTURA)
RESUMEN = RESUMEN.sort_values("auc", ascending=False)
display(RESUMEN[["dataset", "modelo", "protocolo", "accuracy", "acc_baseline", "f1", "auc", "brecha_acc"]].round(3).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(9, 4.8))
colores = ["#C44E52" if d == "B3_DEPSTUDENT" else "#1F5C99" for d in RESUMEN["dataset"][::-1]]
ax.barh(RESUMEN["dataset"][::-1], RESUMEN["auc"][::-1], color=colores)
ax.axvline(0.5, color="#555555", ls="--", label="azar (AUC = 0.5)")
ax.legend(loc="lower right")
ax.set_xlim(0.4, 1.0); ax.set_xlabel("AUC del mejor modelo")
ax.set_title("Capacidad predictiva por dataset (mejor modelo)\nrojo = resultado sospechoso (ver alerta en B3_DEPSTUDENT)")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "resumen_auc_por_dataset.png"), dpi=120); plt.show()

RES["id_dataset"] = RES["dataset"].map(TAREA_DATASET)
RESUMEN["id_dataset"] = RESUMEN["dataset"].map(TAREA_DATASET)     # clave de relacion con vw_dim_dataset
RES.round(4).to_csv(os.path.join(ALMC, "resultados_modelado_completo.csv"), index=False, encoding="utf-8-sig")
RESUMEN.round(4).to_csv(os.path.join(ALMC, "resultados_modelado_resumen.csv"), index=False, encoding="utf-8-sig")
with open(os.path.join(MOD, "manifest_modelos.json"), "w", encoding="utf-8") as f:
    json.dump({"fecha": EJECUCION.strftime("%Y-%m-%d %H:%M:%S"), "booster": BOOST_NAME, "modelos": META},
              f, indent=2, ensure_ascii=False)
print(f"Guardados: {len(META)} modelos (.pkl + .json) en {MOD}")
print("Resultados en:", ALMC)

### Carga de resultados al datawarehouse
Los resultados se guardan en `dwh_modelo_resultados` (script `dwh_estrella.sql`): una fila por dataset × modelo × ejecución, con `id_dataset` como clave foránea hacia `dim_dataset`. Es la **segunda tabla de hechos** de la constelación de consumo; comparte con la estrella la dimensión `vw_dim_dataset`. Si la tabla aún no existe, el paso se omite y los CSV ya están guardados.
Nota: Dreaddit (train + test) se registra con `id_dataset = 8`; sus métricas se miden sobre la partición de prueba.

In [ ]:
# Resultados al DWH: segunda tabla de hechos de la constelacion (grano: dataset x modelo), enlazada a vw_dim_dataset por id_dataset
ALERTAS = {"B3_DEPSTUDENT": "AUC cercano a 1: posible etiqueta generada por formula; consultar con el tutor",
           "B1_ACADSTRESS": "n=140: resultado exploratorio (el Accuracy no supera al baseline)"}
mejor_por_ds = RESUMEN.set_index("dataset")["modelo"].to_dict()
tabla = RES.copy()
tabla["es_mejor_modelo"] = [mejor_por_ds.get(d_) == m_ for d_, m_ in zip(tabla["dataset"], tabla["modelo"])]
tabla["lectura_capa_armonizada"] = tabla["dataset"].map(LECTURA)
tabla["alerta"] = tabla["dataset"].map(ALERTAS)
tabla["fecha_ejecucion"] = EJECUCION
tabla = tabla.rename(columns={"dataset": "tarea_modelado", "n": "n_registros"})[[
    "id_dataset", "tarea_modelado", "modelo", "protocolo", "n_registros", "accuracy", "f1", "auc", "auc_desv",
    "brecha_acc", "es_mejor_modelo", "target", "lectura_capa_armonizada", "alerta", "fecha_ejecucion"]]
if engine is not None and inspect(engine).has_table("dwh_modelo_resultados"):
    tabla.to_sql("dwh_modelo_resultados", engine, if_exists="append", index=False)
    print(f"{len(tabla)} filas cargadas en dwh_modelo_resultados (ejecucion {EJECUCION:%Y-%m-%d %H:%M:%S})")
    q = """SELECT ds.bloque, ds.dataset, r.modelo, r.auc, r.f1
           FROM dwh_modelo_resultados r JOIN vw_dim_dataset ds ON ds.id_dataset = r.id_dataset
           WHERE r.es_mejor_modelo = 1 AND r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_modelo_resultados)
           ORDER BY r.auc DESC"""
    display(pd.read_sql(text(q), engine).round(3))
else:
    print("dwh_modelo_resultados no disponible: ejecuta dwh_estrella.sql en phpMyAdmin y relanza esta celda.")
    print("(Los CSV de resultados ya estan guardados; este paso solo integra los resultados en el DWH.)")

## Conclusiones del Objetivo 2

1. **Hay señal predictiva real, pero moderada.** En las fuentes con señal real el AUC del mejor modelo está entre ≈ 0.73 y ≈ 0.84 y el Accuracy supera siempre al baseline salvo en los dos casos analizados (Anxiety por desbalance y Academic Stress por tamaño). El estrés se puede **anticipar mejor que el azar, no predecir con precisión individual**.
2. **Ningún algoritmo domina.** XGBoost gana 3 tareas, la Regresión Logística 3 (incluido el híbrido de NLP) y Random Forest 2; en varios casos la diferencia está dentro del ruido. En datos tabulares pequeños, el modelo simple es competitivo y menos propenso al sobreajuste.
3. **Los modelos confirman los patrones del Objetivo 1:** presión académica y aislamiento (Bloque 1), patrón de uso y preocupación frente a tiempo de uso (Bloque 2), lenguaje autorreferencial (NLP), antecedentes familiares frente a estrés percibido en la búsqueda de ayuda, y estabilidad emocional (DASS).
4. **El Accuracy por sí solo engaña:** en Anxiety el baseline lo supera y no detecta ningún caso; AUC y F1 lo evidencian.
5. **Un resultado demasiado bueno es una alerta, no un éxito:** B3_DEPSTUDENT (AUC ≈ 0.998) se reporta con cautela.

### Limitaciones
- Tres tareas con n < 600 (estrés académico, MH Survey, SMMH) → alta varianza; las diferencias entre modelos no son concluyentes.
- Datasets de poblaciones distintas a la estudiantil (jugadores online, usuarios de Reddit, población general en DASS y chanchal24).
- No se aplicó deep learning (Días 9-10); no se afinaron hiperparámetros (búsqueda en rejilla) ni umbrales de decisión.
- Los modelos son de **investigación, no de diagnóstico clínico**.

### Siguiente paso — Entrega 3
Los resultados se consumen en Power BI desde `vw_modelo_resultados` (segunda tabla de hechos de la constelación, enlazada a `vw_dim_dataset` por `id_dataset`), como página de analítica avanzada junto con los KPIs del datawarehouse.